In [25]:
from os import path
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.multiprocessing as mp
import yfinance as yf
from finetuning.finetuning_torch import FinetuningConfig, TimesFMFinetuner
from huggingface_hub import snapshot_download
from torch.utils.data import Dataset
from itertools import product
from timesfm import TimesFm, TimesFmCheckpoint, TimesFmHparams
from timesfm.pytorch_patched_decoder import PatchedTimeSeriesDecoder
import os
import json
import matplotlib.pyplot as plt
from timesfm_functions import calculate_prediction_metrics
import matplotlib.pyplot as plt

In [26]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.dates as mdates
# from matplotlib.ticker import MaxNLocator
# from statsmodels.graphics.tsaplots import plot_pacf
# import os
# from pathlib import Path

# # -------------------------
# # Helpers for metrics
# # -------------------------
# def _as_1d(a):
#     a = np.asarray(a)
#     if a.ndim > 1:
#         a = a.reshape(-1)
#     return a

# def _align_and_mask(y_true, y_pred, for_mape=False):
#     yt = _as_1d(y_true)
#     yp = _as_1d(y_pred)
#     n = min(len(yt), len(yp))
#     yt, yp = yt[:n], yp[:n]
#     m = np.isfinite(yt) & np.isfinite(yp)
#     if for_mape:
#         m &= (yt != 0)
#     return yt, yp, m

# # -------------------------
# # Plotting
# # -------------------------
# def line_plot(dates, values, ylabel, graphtitle='Time Series', linecolor='blue', ax=None, show=True, useDates=False):
#     """
#     Plots a time series line chart, handling both integer x and datetime x gracefully.
#     """
#     new_plot = ax is None
#     fig = None
#     if new_plot:
#         fig, ax = plt.subplots(figsize=(14, 6))

#     values = np.asarray(values, dtype=float)
#     if useDates:
#         dates = pd.to_datetime(dates)
#         ax.plot(dates, values, label=ylabel, color=linecolor)
#         ax.set_xlabel('Date')
#         # Adaptive tick locator
#         span_days = (dates.max() - dates.min()).days if len(dates) else 0
#         if span_days >= 730:
#             ax.xaxis.set_major_locator(mdates.YearLocator())
#             ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
#         elif span_days >= 120:
#             ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
#             ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
#         else:
#             ax.xaxis.set_major_locator(mdates.AutoDateLocator())
#             ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
#         ax.tick_params(axis='x', rotation=45)
#     else:
#         x = np.arange(len(values))
#         ax.plot(x, values, label=ylabel, color=linecolor)
#         ax.set_xlabel('trading day')
#         ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))

#     ax.set_ylabel(ylabel)
#     ax.grid(True, alpha=0.3)
#     ax.set_title(graphtitle)
#     ax.legend()

#     if show and new_plot:
#         plt.tight_layout()
#         plt.show()

#     return ax, fig

# # -------------------------
# # Metrics (NaN/Inf safe)
# # -------------------------
# def mase(y_true, y_pred):
#     """
#     Mean Absolute Scaled Error (MASE), robust to NaN/Inf and constant series.
#     Scales by the mean absolute difference of y_true (naive forecast).
#     """
#     yt, yp, m = _align_and_mask(y_true, y_pred)
#     if not np.any(m):
#         return np.nan
#     abs_err = np.abs(yt[m] - yp[m]).mean()

#     # Scale from y_true diffs (drop NaN/Inf and require at least 2 points)
#     yt_clean = _as_1d(y_true)
#     yt_clean = yt_clean[np.isfinite(yt_clean)]
#     if yt_clean.size < 2:
#         return np.nan
#     denom = np.abs(np.diff(yt_clean)).mean()
#     if not np.isfinite(denom) or denom == 0:
#         return np.nan
#     return abs_err / denom

# def rmse(y_true, y_pred):
#     """
#     Root Mean Squared Error (RMSE), NaN/Inf-safe.
#     """
#     yt, yp, m = _align_and_mask(y_true, y_pred)
#     if not np.any(m):
#         return np.nan
#     d = yt[m] - yp[m]
#     return float(np.sqrt(np.mean(d * d)))

# def mse(y_true, y_pred):
#     """
#     Mean Squared Error (MSE), NaN/Inf-safe.
#     """
#     yt, yp, m = _align_and_mask(y_true, y_pred)
#     if not np.any(m):
#         return np.nan
#     d = yt[m] - yp[m]
#     return float(np.mean(d * d))

# def mae(y_true, y_pred):
#     """
#     Mean Absolute Error (MAE), NaN/Inf-safe.
#     """
#     yt, yp, m = _align_and_mask(y_true, y_pred)
#     if not np.any(m):
#         return np.nan
#     return float(np.mean(np.abs(yt[m] - yp[m])))

# def mape(y_true, y_pred):
#     """
#     Mean Absolute Percentage Error (MAPE in %), excludes pairs where y_true == 0 and any NaN/Inf.
#     """
#     yt, yp, m = _align_and_mask(y_true, y_pred, for_mape=True)
#     if not np.any(m):
#         return np.nan
#     return float(np.mean(np.abs((yt[m] - yp[m]) / yt[m])) * 100.0)

# # -------------------------
# # Volatility
# # -------------------------
# def compute_daily_volatility(prices: pd.Series, window: int = 20, method: str = 'rolling', returns: str = 'log') -> pd.Series:
#     """
#     Estimate annualized daily volatility from *returns* using rolling or exponential std.
#     - prices: pd.Series indexed by datetime
#     - returns: 'log' or 'simple'
#     """
#     if not isinstance(prices, pd.Series):
#         prices = pd.Series(prices)

#     prices = prices.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
#     if returns not in ('log', 'simple'):
#         raise ValueError("returns must be 'log' or 'simple'.")

#     if returns == 'log':
#         rets = np.log(prices).diff()
#     else:
#         rets = prices.pct_change()

#     rets = rets.replace([np.inf, -np.inf], np.nan)

#     if method == 'rolling':
#         vol = rets.rolling(window=window, min_periods=window).std()
#     elif method == 'ewm':
#         vol = rets.ewm(span=window, adjust=False).std()
#     else:
#         raise ValueError("method must be either 'rolling' or 'ewm'.")

#     return vol * np.sqrt(252.0)

# # -------------------------
# # Expiry helpers
# # -------------------------
# def pred_char_to_value(s):
#     mapping = {'1w': 5, '1m': 22, '3m': 66, '1y': 252}
#     if s not in mapping:
#         raise ValueError(f"Unknown horizon '{s}'. Expected one of {list(mapping)}.")
#     return mapping[s]

# def pred_value_to_char(n):
#     mapping = {5: '1w', 22: '1m', 66: '3m', 252: '1y'}
#     if n not in mapping:
#         raise ValueError(f"Unknown horizon value '{n}'. Expected one of {list(mapping)}.")
#     return mapping[n]

# # -------------------------
# # IO & PACF
# # -------------------------
# def load_data(data_path='../Data/aluminium_pre_inputs.csv', index_col=0, parse_dates=True):
#     """
#     Robust CSV loader. If index appears to be date-like, parse it to DatetimeIndex.
#     """
#     data_path = Path(data_path)
#     if not data_path.exists():
#         raise FileNotFoundError(f"File not found: {data_path}")

#     df = pd.read_csv(data_path, index_col=index_col)
#     if parse_dates:
#         # Try to parse index if it's not numeric
#         try:
#             if not np.issubdtype(df.index.dtype, np.number):
#                 df.index = pd.to_datetime(df.index, errors='ignore')
#         except Exception:
#             pass
#         # Also parse a 'date' column if present
#         if 'date' in df.columns and not np.issubdtype(df['date'].dtype, np.datetime64):
#             df['date'] = pd.to_datetime(df['date'], errors='coerce')
#     return df

# def plot_squared_pacf(df, col_name, lags=40, alpha=0.05, plots_dir="plots", show=False):
#     """
#     Plot and save PACF of the *squared* series for the given column.
#     Returns the saved file path.
#     """
#     if col_name not in df.columns:
#         raise KeyError(f"Column '{col_name}' not in dataframe. Available: {list(df.columns)}")

#     series = pd.Series(df[col_name], dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
#     sq = (series ** 2).dropna()

#     plots_path = Path(plots_dir)
#     plots_path.mkdir(parents=True, exist_ok=True)

#     fig, ax = plt.subplots(figsize=(10, 4))
#     plot_pacf(sq, lags=lags, alpha=alpha, method='ywm', ax=ax)
#     ax.set_title(f"PACF of squared {col_name}")
#     ax.grid(True, alpha=0.3)

#     filename = plots_path / f"pacf_squared_{col_name}.png"
#     fig.savefig(filename, dpi=300, bbox_inches='tight')
#     if show:
#         plt.show()
#     plt.close(fig)
#     print(f"Plot saved as: {filename}")
#     return str(filename)

# # -------------------------
# # Feature selection helper
# # -------------------------
# def find_n_best_features(expiry, n, corr_filename='absolute_feature_correlations.csv'):
#     """
#     Select top-n features by correlation with volatility for the given expiry.
#     Looks for Feature_selection/<corr_filename> relative to this file.
#     """
#     try:
#         expiry_tag = pred_value_to_char(expiry)
#     except ValueError as e:
#         raise

#     current_file_dir = Path(__file__).resolve().parent
#     feature_selection_dir = current_file_dir / 'Feature_selection'
#     corr_file = feature_selection_dir / corr_filename

#     if not corr_file.exists():
#         raise FileNotFoundError(f"Correlation file not found: {corr_file}")

#     best_features = pd.read_csv(corr_file, index_col=0)
#     target_col = f'{expiry_tag}_exp'
#     if target_col not in best_features.columns:
#         raise KeyError(
#             f"Column '{target_col}' not in {corr_file.name}. "
#             f"Available columns: {list(best_features.columns)}"
#         )

#     # Sort descending by correlation (assumed absolute already per filename)
#     top_rows = best_features.sort_values(by=target_col, ascending=False).head(int(n))
#     return top_rows.index.tolist()


In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
from statsmodels.graphics.tsaplots import plot_pacf
import os
from pathlib import Path

# -------------------------
# Helpers for metrics
# -------------------------
def _as_1d(a):
    a = np.asarray(a)
    if a.ndim > 1:
        a = a.reshape(-1)
    return a

def _align_and_mask(y_true, y_pred, for_mape=False):
    yt = _as_1d(y_true)
    yp = _as_1d(y_pred)
    n = min(len(yt), len(yp))
    yt, yp = yt[:n], yp[:n]
    m = np.isfinite(yt) & np.isfinite(yp)
    if for_mape:
        m &= (yt != 0)
    return yt, yp, m

# -------------------------
# Plotting
# -------------------------
def line_plot(dates, values, ylabel, graphtitle='Time Series', linecolor='blue', ax=None, show=True, useDates=False):
    """
    Plots a time series line chart, handling both integer x and datetime x gracefully.
    """
    new_plot = ax is None
    fig = None
    if new_plot:
        fig, ax = plt.subplots(figsize=(14, 6))

    values = np.asarray(values, dtype=float)
    if useDates:
        dates = pd.to_datetime(dates)
        ax.plot(dates, values, label=ylabel, color=linecolor)
        ax.set_xlabel('Date')
        # Adaptive tick locator
        span_days = (dates.max() - dates.min()).days if len(dates) else 0
        if span_days >= 730:
            ax.xaxis.set_major_locator(mdates.YearLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        elif span_days >= 120:
            ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        else:
            ax.xaxis.set_major_locator(mdates.AutoDateLocator())
            ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        ax.tick_params(axis='x', rotation=45)
    else:
        x = np.arange(len(values))
        ax.plot(x, values, label=ylabel, color=linecolor)
        ax.set_xlabel('trading day')
        ax.xaxis.set_major_locator(MaxNLocator(nbins=10, integer=True))

    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    ax.set_title(graphtitle)
    ax.legend()

    if show and new_plot:
        plt.tight_layout()
        plt.show()

    return ax, fig

# -------------------------
# Metrics (NaN/Inf safe)
# -------------------------
def mase(y_true, y_pred):
    """
    Mean Absolute Scaled Error (MASE), robust to NaN/Inf and constant series.
    Scales by the mean absolute difference of y_true (naive forecast).
    """
    yt, yp, m = _align_and_mask(y_true, y_pred)
    if not np.any(m):
        return np.nan
    abs_err = np.abs(yt[m] - yp[m]).mean()

    # Scale from y_true diffs (drop NaN/Inf and require at least 2 points)
    yt_clean = _as_1d(y_true)
    yt_clean = yt_clean[np.isfinite(yt_clean)]
    if yt_clean.size < 2:
        return np.nan
    denom = np.abs(np.diff(yt_clean)).mean()
    if not np.isfinite(denom) or denom == 0:
        return np.nan
    return abs_err / denom

def rmse(y_true, y_pred):
    """
    Root Mean Squared Error (RMSE), NaN/Inf-safe.
    """
    yt, yp, m = _align_and_mask(y_true, y_pred)
    if not np.any(m):
        return np.nan
    d = yt[m] - yp[m]
    return float(np.sqrt(np.mean(d * d)))

def mse(y_true, y_pred):
    """
    Mean Squared Error (MSE), NaN/Inf-safe.
    """
    yt, yp, m = _align_and_mask(y_true, y_pred)
    if not np.any(m):
        return np.nan
    d = yt[m] - yp[m]
    return float(np.mean(d * d))

def mae(y_true, y_pred):
    """
    Mean Absolute Error (MAE), NaN/Inf-safe.
    """
    yt, yp, m = _align_and_mask(y_true, y_pred)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs(yt[m] - yp[m])))

def mape(y_true, y_pred):
    """
    Mean Absolute Percentage Error (MAPE in %), excludes pairs where y_true == 0 and any NaN/Inf.
    """
    yt, yp, m = _align_and_mask(y_true, y_pred, for_mape=True)
    if not np.any(m):
        return np.nan
    return float(np.mean(np.abs((yt[m] - yp[m]) / yt[m])) * 100.0)

# -------------------------
# Volatility
# -------------------------
def compute_daily_volatility(prices: pd.Series, window: int = 20, method: str = 'rolling', returns: str = 'log') -> pd.Series:
    """
    Estimate annualized daily volatility from *returns* using rolling or exponential std.
    - prices: pd.Series indexed by datetime
    - returns: 'log' or 'simple'
    """
    if not isinstance(prices, pd.Series):
        prices = pd.Series(prices)

    prices = prices.astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    if returns not in ('log', 'simple'):
        raise ValueError("returns must be 'log' or 'simple'.")

    if returns == 'log':
        rets = np.log(prices).diff()
    else:
        rets = prices.pct_change()

    rets = rets.replace([np.inf, -np.inf], np.nan)

    if method == 'rolling':
        vol = rets.rolling(window=window, min_periods=window).std()
    elif method == 'ewm':
        vol = rets.ewm(span=window, adjust=False).std()
    else:
        raise ValueError("method must be either 'rolling' or 'ewm'.")

    return vol * np.sqrt(252.0)

# -------------------------
# Expiry helpers
# -------------------------
def pred_char_to_value(s):
    mapping = {'1w': 5, '1m': 22, '3m': 66, '1y': 252}
    if s not in mapping:
        raise ValueError(f"Unknown horizon '{s}'. Expected one of {list(mapping)}.")
    return mapping[s]

def pred_value_to_char(n):
    mapping = {5: '1w', 22: '1m', 66: '3m', 252: '1y'}
    if n not in mapping:
        raise ValueError(f"Unknown horizon value '{n}'. Expected one of {list(mapping)}.")
    return mapping[n]

# -------------------------
# IO & PACF
# -------------------------
def load_data(data_path='../Data/aluminium_pre_inputs.csv', index_col=0, parse_dates=True):
    """
    Robust CSV loader. If index appears to be date-like, parse it to DatetimeIndex.
    """
    data_path = Path(data_path)
    if not data_path.exists():
        raise FileNotFoundError(f"File not found: {data_path}")

    df = pd.read_csv(data_path, index_col=index_col)
    if parse_dates:
        # Try to parse index if it's not numeric
        try:
            if not np.issubdtype(df.index.dtype, np.number):
                df.index = pd.to_datetime(df.index, errors='ignore')
        except Exception:
            pass
        # Also parse a 'date' column if present
        if 'date' in df.columns and not np.issubdtype(df['date'].dtype, np.datetime64):
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df

def plot_squared_pacf(df, col_name, lags=40, alpha=0.05, plots_dir="plots", show=False):
    """
    Plot and save PACF of the *squared* series for the given column.
    Returns the saved file path.
    """
    if col_name not in df.columns:
        raise KeyError(f"Column '{col_name}' not in dataframe. Available: {list(df.columns)}")

    series = pd.Series(df[col_name], dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    sq = (series ** 2).dropna()

    plots_path = Path(plots_dir)
    plots_path.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(10, 4))
    plot_pacf(sq, lags=lags, alpha=alpha, method='ywm', ax=ax)
    ax.set_title(f"PACF of squared {col_name}")
    ax.grid(True, alpha=0.3)

    filename = plots_path / f"pacf_squared_{col_name}.png"
    fig.savefig(filename, dpi=300, bbox_inches='tight')
    if show:
        plt.show()
    plt.close(fig)
    print(f"Plot saved as: {filename}")
    return str(filename)

# -------------------------
# Feature selection helper
# -------------------------
def find_n_best_features(expiry, n, corr_filename='absolute_feature_correlations.csv'):
    """
    Select top-n features by correlation with volatility for the given expiry.
    Looks for Feature_selection/<corr_filename> relative to this file.
    """
    try:
        expiry_tag = pred_value_to_char(expiry)
    except ValueError as e:
        raise

    current_file_dir = Path(__file__).resolve().parent
    feature_selection_dir = current_file_dir / 'Feature_selection'
    corr_file = feature_selection_dir / corr_filename

    if not corr_file.exists():
        raise FileNotFoundError(f"Correlation file not found: {corr_file}")

    best_features = pd.read_csv(corr_file, index_col=0)
    target_col = f'{expiry_tag}_exp'
    if target_col not in best_features.columns:
        raise KeyError(
            f"Column '{target_col}' not in {corr_file.name}. "
            f"Available columns: {list(best_features.columns)}"
        )

    # Sort descending by correlation (assumed absolute already per filename)
    top_rows = best_features.sort_values(by=target_col, ascending=False).head(int(n))
    return top_rows.index.tolist()


In [28]:
class TimeSeriesDataset(Dataset):
  """Dataset for time series data compatible with TimesFM."""

  def __init__(self,
               series: np.ndarray,
               context_length: int,
               horizon_length: int,
               freq_type: int = 0):
    """
        Initialize dataset.

        Args:
            series: Time series data
            context_length: Number of past timesteps to use as input
            horizon_length: Number of future timesteps to predict
            freq_type: Frequency type (0, 1, or 2)
        """
    if freq_type not in [0, 1, 2]:
      raise ValueError("freq_type must be 0, 1, or 2")

    self.series = series
    self.context_length = context_length
    self.horizon_length = horizon_length
    self.freq_type = freq_type
    self._prepare_samples()

  def _prepare_samples(self) -> None:
    """Prepare sliding window samples from the time series."""
    self.samples = []
    total_length = self.context_length + self.horizon_length

    for start_idx in range(0, len(self.series) - total_length + 1):
      end_idx = start_idx + self.context_length
      x_context = self.series[start_idx:end_idx]
      x_future = self.series[end_idx:end_idx + self.horizon_length]
      self.samples.append((x_context, x_future))

  def __len__(self) -> int:
    return len(self.samples)

  def __getitem__(
      self, index: int
  ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    x_context, x_future = self.samples[index]

    x_context = torch.tensor(x_context, dtype=torch.float32)
    x_future = torch.tensor(x_future, dtype=torch.float32)

    input_padding = torch.zeros_like(x_context)
    freq = torch.tensor([self.freq_type], dtype=torch.long)

    return x_context, input_padding, freq, x_future

def prepare_datasets(series, context_length, horizon_length, freq_type=0, train_split=0.8):
  """
    Prepare training and validation datasets from time series data.

    Args:
        series: Input time series data
        context_length: Number of past timesteps to use
        horizon_length: Number of future timesteps to predict
        freq_type: Frequency type (0, 1, or 2)
        train_split: Fraction of data to use for training

    Returns:
        Tuple of (train_dataset, val_dataset)
    """
  train_size = int(len(series) * train_split)
  train_data = series[:train_size]
  val_data = series[train_size:]

  # Create datasets with specified frequency type
  train_dataset = TimeSeriesDataset(train_data, context_length=context_length, horizon_length=horizon_length, freq_type=freq_type)
  val_dataset = TimeSeriesDataset(val_data, context_length=context_length, horizon_length=horizon_length, freq_type=freq_type)

  return train_dataset, val_dataset

In [29]:
def get_model(horizon_len, use_positional_embedding, context_len, load_weights=False):
	device = "cuda" if torch.cuda.is_available() else "cpu"
	repo_id = "google/timesfm-2.0-500m-pytorch"
	hparams = TimesFmHparams(
		backend=device,
		per_core_batch_size=32,
		horizon_len=horizon_len,
		num_layers=50,
		use_positional_embedding=use_positional_embedding,
		context_len=context_len, # multiples of 32
	)
	tfm = TimesFm(hparams=hparams,checkpoint=TimesFmCheckpoint(huggingface_repo_id=repo_id))
 
	model = PatchedTimeSeriesDecoder(tfm._model_config)
	if load_weights:
		checkpoint_path = path.join(snapshot_download(repo_id), "torch_model.ckpt")
		loaded_checkpoint = torch.load(checkpoint_path, weights_only=True)
		model.load_state_dict(loaded_checkpoint)
	return model, hparams, tfm._model_config


In [30]:
def plot_predictions(model, val_dataset, save_path="predictions.png"):
	"""
		Plot model predictions against ground truth for a batch of validation data.

		Args:
			model: Trained TimesFM model
			val_dataset: Validation dataset
			save_path: Path to save the plot
		"""
	model.eval()

	x_context, x_padding, freq, x_future = val_dataset[0]
	x_context = x_context.unsqueeze(0)  # Add batch dimension
	x_padding = x_padding.unsqueeze(0)
	freq = freq.unsqueeze(0)
	x_future = x_future.unsqueeze(0)

	device = next(model.parameters()).device
	x_context = x_context.to(device)
	x_padding = x_padding.to(device)
	freq = freq.to(device)
	x_future = x_future.to(device)

	with torch.no_grad():
		predictions = model(x_context, x_padding.float(), freq)
		predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
		last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

	context_vals = x_context[0].cpu().numpy()
	future_vals = x_future[0].cpu().numpy()
	pred_vals = last_patch_pred[0].cpu().numpy()

	context_len = len(context_vals)
	horizon_len = len(future_vals)

	plt.figure(figsize=(12, 6))

	plt.plot(range(context_len),
				context_vals,
				label="Historical Data",
				color="blue",
				linewidth=2)

	plt.plot(
		range(context_len, context_len + horizon_len),
		future_vals,
		label="Ground Truth",
		color="green",
		linestyle="--",
		linewidth=2,
	)

	plt.plot(range(context_len, context_len + horizon_len),
				pred_vals,
				label="Prediction",
				color="red",
				linewidth=2)

	plt.xlabel("Time Step")
	plt.ylabel("Value")
	plt.title("TimesFM Predictions vs Ground Truth")
	plt.legend()
	plt.grid(True)

	if save_path:
		plt.savefig(save_path)
		print(f"Plot saved to {save_path}")

	plt.close()


In [31]:
def get_data(context_len, horizon_len, freq_type, expiry):
	df = pd.read_csv('aluminium_pre_inputs.csv')
	time_series = df[f'{pred_value_to_char(expiry)}_vol'].dropna().values

	train_dataset, val_dataset = prepare_datasets(
		series=time_series,
		context_length=context_len,
		horizon_length=horizon_len,
		freq_type=freq_type,
		train_split=0.8,
	)

	print(f"Created datasets:")
	print(f"- Training samples: {len(train_dataset)}")
	print(f"- Validation samples: {len(val_dataset)}")
	print(f"- Using frequency type: {freq_type}")
	return train_dataset, val_dataset, len(val_dataset)

In [32]:
hMap = {
    5: { 'freq': 0, 'pos': 0},
    22: { 'freq': 1, 'pos': 0},
    66: { 'freq': 0, 'pos': 1},
    252: { 'freq': 1, 'pos': 0},
}

In [33]:
# expiries = [5, 22, 66, 252]
# models = []

# for expiry in expiries:
#     """Basic example of finetuning TimesFM on stock data."""
#     model, hparams, tfm_config = get_model(load_weights=True)
#     config = FinetuningConfig(batch_size=256,
#                             num_epochs=10,
#                             learning_rate=1e-4,
#                             use_wandb=True,
#                             freq_type=hMap[expiry]['freq'],
#                             log_every_n_steps=20,
#                             val_check_interval=0.5,
#                             use_quantile_loss=True)

#     train_dataset, val_dataset = get_data(context_len=128, horizon_len=tfm_config.horizon_len, freq_type=config.freq_type, expiry=expiry)
#     finetuner = TimesFMFinetuner(model, config)

#     print("\nStarting finetuning...")
#     results = finetuner.finetune(train_dataset=train_dataset,
#                                 val_dataset=val_dataset)
#     models.append(model)
#     print("\nFinetuning completed!")
#     print(f"Training history: {len(results['history']['train_loss'])} epochs")

In [34]:
def get_pred_vals(idx, model, val_dataset, horizon_len):
    model.eval()

    x_context, x_padding, freq, x_future = val_dataset[idx]
    x_context = x_context.unsqueeze(0)  # Add batch dimension
    x_padding = x_padding.unsqueeze(0)
    freq = freq.unsqueeze(0)
    x_future = x_future.unsqueeze(0)

    device = next(model.parameters()).device
    x_context = x_context.to(device)
    x_padding = x_padding.to(device)
    freq = freq.to(device)
    x_future = x_future.to(device)

    with torch.no_grad():
        predictions = model(x_context, x_padding.float(), freq)
        predictions_mean = predictions[..., 0]  # [B, N, horizon_len]
        last_patch_pred = predictions_mean[:, -1, :]  # [B, horizon_len]

    context_vals = x_context[0].cpu().numpy()
    future_vals = x_future[0].cpu().numpy()
    pred_vals = last_patch_pred[0].cpu().numpy()
    return pred_vals[horizon_len-1], future_vals[horizon_len-1]

In [35]:
# context_len = 64
# horizon_len = 5
# use_positional_embedding = False

# """Basic example of finetuning TimesFM on stock data."""
# model, hparams, tfm_config = get_model(horizon_len, use_positional_embedding, context_len, load_weights=False)
# config = FinetuningConfig(batch_size=64,
#                         num_epochs=5,
#                         learning_rate=1e-4,
#                         use_wandb=True,
#                         freq_type=hMap[horizon_len]['freq'],
#                         log_every_n_steps=20,
#                         val_check_interval=0.5,
#                         use_quantile_loss=True)
# # train_dataset, val_dataset = get_data(context_len=context_len, horizon_len=horizon_len, freq_type=freq_type, expiry=horizon_len)
# train_dataset, val_dataset, len_val_data = get_data(context_len=context_len, horizon_len=tfm_config.horizon_len, freq_type=config.freq_type, expiry=horizon_len)
# finetuner = TimesFMFinetuner(model, config)

# print("\nStarting finetuning...")
# results = finetuner.finetune(train_dataset=train_dataset, val_dataset=val_dataset)
# print("\nFinetuning completed!")
# print(f"Training history: {len(results['history']['train_loss'])} epochs")


# preds = []
# true_test = []
# for i in range(len_val_data):
#     pred_i, true_i = get_pred_vals(idx=i, model=model, val_dataset=val_dataset, horizon_len=horizon_len)
#     preds.append(pred_i)
#     true_test.append(true_i)

# ax, fig = line_plot(preds, preds, 'pred', show=False)
# _, _ = line_plot(true_test, true_test, 'true', linecolor='red', ax=ax, show=False)

# calculate_prediction_metrics(preds, true_test)

In [36]:
def make_json_safe(obj):
    """Recursively convert NumPy/other non-JSON types to native Python types."""
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    return obj

In [37]:
context_len_list  = [32, 64, 128]
horizon_len_list  = [5, 22, 66]
freq = [0, 1, 2]
use_positional_embedding = False

# --- output dirs ---
BASE_DIR     = "timesfm_finetuned_results"
METRICS_DIR  = os.path.join(BASE_DIR, "metrics")
PLOTS_DIR    = os.path.join(BASE_DIR, "plots")
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

for c_len, h_len, f in product(context_len_list, horizon_len_list, freq):
    print(f"\n=== Run: context_len={c_len}, horizon_len={h_len} ===")
    if c_len == 256 and h_len == 66: continue
    if c_len < h_len: continue
    # Build model/config for this pair
    # If your get_model expects a single int for horizon_len, pass h_len.
    # If it expects a list, change to [h_len].
    model, hparams, tfm_config = get_model(
        horizon_len=h_len,
        use_positional_embedding=use_positional_embedding,
        context_len=c_len,
        load_weights=False
    )

    config = FinetuningConfig(
        batch_size=64,
        num_epochs=5,
        learning_rate=1e-4,
        use_wandb=True,
        freq_type=f,
        log_every_n_steps=20,
        val_check_interval=0.5,
        use_quantile_loss=True
    )

    # If your data loader expects horizon_len from tfm_config, use tfm_config.horizon_len.
    # Otherwise explicitly use h_len for both.
    train_dataset, val_dataset, len_val_data = get_data(
        context_len=c_len,
        horizon_len=tfm_config.horizon_len,
        freq_type=config.freq_type,
        expiry=h_len
    )

    finetuner = TimesFMFinetuner(model, config)

    print("Starting finetuning...")
    results = finetuner.finetune(train_dataset=train_dataset, val_dataset=val_dataset)
    print("Finetuning completed!")
    print(f"Training history: {len(results['history']['train_loss'])} epochs")

    # Collect predictions vs truth on the validation split
    preds, true_test = [], []
    for i in range(len_val_data):
        pred_i, true_i = get_pred_vals(
            idx=i,
            model=model,
            val_dataset=val_dataset,
            horizon_len=h_len
        )
        preds.append(pred_i)
        true_test.append(true_i)

    # Plot predictions (blue) and truth (red), and save figure
    ax, fig = line_plot(preds, preds, 'pred', show=False)
    _, _ = line_plot(true_test, true_test, 'true', linecolor='red', ax=ax, show=False)

    plot_path = os.path.join(PLOTS_DIR, f"timesfm_pred_vs_true_h{h_len}_c{c_len}_5epochs_f{f}.png")
    fig.savefig(plot_path, bbox_inches="tight", dpi=150)
    plt.close(fig)

    metrics = calculate_prediction_metrics(preds, true_test)

    # make metrics JSON-serializable
    metrics_safe = make_json_safe(metrics)

    metrics_path = os.path.join(METRICS_DIR, f"metrics_h{h_len}_c{c_len}_5epochs_f{f}.json")
    with open(metrics_path, "w") as file:
        json.dump(metrics_safe, file, indent=2)


    print(f"Saved plot -> {plot_path}")
    print(f"Saved metrics -> {metrics_path}")


    # --- Save predictions and ground truth data ---
    data_dict = {
        "preds": make_json_safe(preds),
        "true": make_json_safe(true_test)
    }

    data_path = os.path.join(
        METRICS_DIR,
        f"data_h{h_len}_c{c_len}_5epochs_f{f}.json"
    )
    with open(data_path, "w") as f:
        json.dump(data_dict, f, indent=2)

    print(f"Saved data -> {data_path}")



=== Run: context_len=32, horizon_len=5 ===


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 32974.09it/s]


Created datasets:
- Training samples: 1793
- Validation samples: 330
- Using frequency type: 0


Starting finetuning...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


...

Created datasets:
- Training samples: 1649
- Validation samples: 221
- Using frequency type: 2


Starting finetuning...


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▃▅▆█
learning_rate,▁▁▁▁▁
train_loss,█▃▂▁▁
val_loss,█▂▁▂▃
epoch,5
learning_rate,0.0001
train_loss,0.27872
val_loss,0.29964


Finetuning completed!
Training history: 5 epochs
Saved plot -> timesfm_finetuned_results/plots/timesfm_pred_vs_true_h66_c128_5epochs_f2.png
Saved metrics -> timesfm_finetuned_results/metrics/metrics_h66_c128_5epochs_f2.json
Saved data -> timesfm_finetuned_results/metrics/data_h66_c128_5epochs_f2.json
